**Imports and setup.** Imports TensorFlow and the batch-prediction processor, and loads `config.yaml` to locate the trained model bundle.

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TensorFlow INFO and WARNING messages

import numpy as np
import pickle
from pathlib import Path

import yaml
from upwins_veg import batch_predict as processor

# config.yaml (and the paths inside it) are relative to the repo root, but
# this notebook lives in notebooks/. Locate the repo root and resolve every
# configured path against it so it works from either directory.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'config.yaml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

with open(REPO_ROOT / 'config.yaml') as _f:
    CONFIG = yaml.safe_load(_f)
for _section in ('paths', 'prediction'):
    for _key, _val in CONFIG.get(_section, {}).items():
        if isinstance(_val, str):
            CONFIG[_section][_key] = str(REPO_ROOT / _val)
MODEL_DIR = CONFIG['paths']['model_dir']

import tensorflow as tf
#from tensorflow import keras
#from tensorflow.keras import layers

from sklearn.preprocessing import StandardScaler
import pandas as pd

print(f"Using TensorFlow version: {tf.__version__}")
# Optional: Configure GPU memory growth if needed
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Configured memory growth for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(e)


**Load the model and scaler.** Loads the trained model and its `StandardScaler` from the bundle produced by `01_train_multitask_cnn.ipynb`.

In [ ]:
# Load the trained model bundle produced by 01_train_multitask_cnn.ipynb.
model_path      = os.path.join(MODEL_DIR, 'model.keras')
scaler_filename = os.path.join(MODEL_DIR, 'scaler.pkl')

with open(scaler_filename, 'rb') as file:
    loaded_scaler = pickle.load(file)

model = tf.keras.models.load_model(model_path)


**Load the label maps.** Reads the index→class-name maps from `label_maps.json` and lists the five prediction tasks.

In [ ]:
# Label maps map each model output index back to a human-readable class name.
import json
with open(os.path.join(MODEL_DIR, 'label_maps.json')) as f:
    label_maps = json.load(f)

TASK_NAMES = ['plant', 'age', 'part', 'health', 'lifecycle']


**Preview the label maps.** Prints the label maps so you can confirm the class names the model will output.

In [ ]:
print(label_maps)

**Classify the image.** Reads the image and output paths from the config, verifies the image's bands match what the model was trained on (a hard stop on mismatch), then classifies the image chunk-by-chunk and writes an ENVI classification map per task to the output directory.

In [ ]:
import numpy as np
import json
import spectral

INPUT_FILE_PATH  = CONFIG['prediction']['input_hdr']
OUTPUT_DIRECTORY = CONFIG['prediction']['output_dir']
ROWS_PER_CHUNK   = CONFIG['prediction']['rows_per_chunk']  # tune to available RAM

# --- Verify the image bands match what the model was trained on ----------
# A wrong or differently-resampled image would otherwise produce confident
# nonsense. Fail loudly on a band-count mismatch; warn on a centers drift.
with open(os.path.join(MODEL_DIR, 'wavelengths.json')) as f:
    expected_wl = np.asarray(json.load(f), dtype=float)

_im = spectral.envi.open(INPUT_FILE_PATH)
img_wl = np.asarray(_im.bands.centers, dtype=float)
assert img_wl.shape == expected_wl.shape, (
    f"Band-count mismatch: image has {img_wl.shape[0]} bands, "
    f"model expects {expected_wl.shape[0]}. Resample the image first.")
if not np.allclose(img_wl, expected_wl, atol=1.0):
    print("WARNING: image band centers differ from the model's by >1 nm.")

# Run classification, writing an ENVI classification image to OUTPUT_DIRECTORY.
processor.batch_classify(
    INPUT_FILE_PATH, OUTPUT_DIRECTORY, model, loaded_scaler,
    label_maps, TASK_NAMES, ROWS_PER_CHUNK)
